# Session 2 - Multi-Agent Experiment Loop

Goal: connect Experimenter Agent and Subject LLM into a controlled batch-run pipeline.

## Learning Objectives

1. Build prompt packages from stimuli.
2. Wrap LLM calls in a consistent response schema.
3. Run many trials and persist raw outputs.

## TODO Mapping to Source Files

Complete these modules after this notebook:

| File | Role |
|---|---|
| `src/experiments/config.py` | Config schema, validation, and default values |
| `src/agents/experimenter.py` | Build structured prompt packages from stimulus records |
| `src/agents/subject_llm.py` | LLM call wrapper and response normalization |
| `src/agents/dialogue_runner.py` | Per-trial and batch execution loop with fault tolerance |
| `src/experiments/runner.py` | End-to-end orchestration and JSONL artifact persistence |

In [ ]:
# Path setup for notebook
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

print("Project root:   ", PROJECT_ROOT)
print("Stimuli input:  ", DATA_DIR / "processed")
print("Run output:     ", ARTIFACTS_DIR / "runs")

## Session 2 Execution Spec

This section answers four practical questions.

### 1. What is the data flow?

```
StimulusRecord
  → build_experimenter_prompt()       # src/agents/experimenter.py
  → attach_run_metadata()             # src/agents/experimenter.py
  → invoke_subject_llm()              # src/agents/subject_llm.py
  → normalize_subject_response()      # src/agents/subject_llm.py
  → run_single_trial()                # src/agents/dialogue_runner.py
  → persist_run_outputs()             # src/experiments/runner.py
  → artifacts/runs/run_<id>.jsonl
```

### 2. What is the input?
- Validated stimuli from Session 1: `data/processed/stimuli_balanced_seed_<seed>.jsonl`
- An experiment config dict (see Config Schema section below)

### 3. How many trials?

| Scale | Num Trials | Purpose |
|---|---|---|
| Smoke test | 10–20 | Verify pipeline runs without errors |
| Pilot | 50–100 | Baseline comparison across conditions |
| Assignment target | 120–240 | Statistical analysis in Session 3 |

Rule: `num_trials` in config must be ≤ `batch_size` used in Session 1.

### 4. Where does output go?
- Run artifact JSONL: `artifacts/runs/run_<run_id>_seed<seed>.jsonl`
- One line per trial; each line is a serialized `TrialOutput` record.

## Config Schema

The experiment config is a plain dictionary with these required fields:

| Field | Type | Example | Purpose |
|---|---|---|---|
| `run_id` | `str` | `"run_001"` | Unique identifier for this run; used in artifact filenames |
| `model_name` | `str` | `"gpt-4o-mini"` | LLM model to call as subject |
| `stimuli_path` | `str` | `"data/processed/stimuli_balanced_seed_42.jsonl"` | Input stimuli from Session 1 |
| `num_trials` | `int` | `80` | How many stimuli to run (must be ≤ total stimuli available) |
| `seed` | `int` | `42` | Random seed for reproducibility |
| `prompt_style` | `str` | `"minimal"` | How the prime is presented: `"minimal"` (prime only) or `"instructed"` (prime + explicit syntax cue) |
| `generation` | `dict` | `{"temperature": 0.3, "max_tokens": 120}` | LLM generation parameters |

All fields are required. `validate_experiment_config()` must return an error for any missing or incorrectly typed field.

In [ ]:
# Example config: full schema with all required fields
example_config = {
    "run_id": "run_001",
    "model_name": "gpt-4o-mini",
    "stimuli_path": "data/processed/stimuli_balanced_seed_42.jsonl",
    "num_trials": 80,
    "seed": 42,
    "prompt_style": "minimal",        # "minimal" or "instructed"
    "generation": {
        "temperature": 0.3,
        "max_tokens": 120,
    },
}

# build_default_config() in src/experiments/config.py should return something like this.
# Students can override specific fields for their own runs.
print("Config keys:", list(example_config.keys()))
print("Generation params:", example_config["generation"])

In [ ]:
# Toy config validation (demonstrates what validate_experiment_config() should check)

REQUIRED_CONFIG_FIELDS = {
    "run_id": str,
    "model_name": str,
    "stimuli_path": str,
    "num_trials": int,
    "seed": int,
    "prompt_style": str,
    "generation": dict,
}

def toy_validate_config(config: dict) -> list:
    errors = []
    for field, expected_type in REQUIRED_CONFIG_FIELDS.items():
        if field not in config:
            errors.append(f"missing required field: '{field}'")
        elif not isinstance(config[field], expected_type):
            errors.append(f"'{field}' must be {expected_type.__name__}, got {type(config[field]).__name__}")
    if "generation" in config and isinstance(config["generation"], dict):
        temp = config["generation"].get("temperature")
        if temp is not None and not (0.0 <= temp <= 1.0):
            errors.append(f"generation.temperature must be in [0, 1], got {temp}")
    if isinstance(config.get("num_trials"), int) and config["num_trials"] <= 0:
        errors.append("num_trials must be > 0")
    return errors

# Test with a valid config
errors = toy_validate_config(example_config)
print("Valid config errors:", len(errors))

# Test with a broken config
broken_config = {"run_id": "run_001", "num_trials": -5, "generation": {"temperature": 2.0}}
broken_errors = toy_validate_config(broken_config)
print(f"\nBroken config errors: {len(broken_errors)}")
for e in broken_errors:
    print(" -", e)

In [ ]:
# Example orchestration pattern only
from src.experiments.runner import execute_experiment

# Will run after TODOs are implemented in src
# summary = execute_experiment(example_config)
# summary

## Session 2 Acceptance Checklist

By the end of Session 2, you should be able to check all of these:

1. `validate_experiment_config(config)` returns zero errors for the example config.
2. `build_experimenter_prompt(stimulus, prompt_style)` returns a dict with `system_prompt` and `user_prompt` keys.
3. `attach_run_metadata(package, run_id, trial_index)` adds `run_id` and `trial_index` to the package.
4. `invoke_subject_llm()` calls the LLM and returns a raw response without crashing.
5. `normalize_subject_response(raw)` maps the provider response to the standard `{"text", "finish_reason", "usage"}` shape.
6. `run_single_trial(stimulus, model_name, gen_config)` completes end-to-end and returns a `TrialOutput`.
7. `run_trial_batch(stimuli, ...)` completes 10–20 trials without halting on a single failure.
8. A JSONL artifact exists in `artifacts/runs/` after `persist_run_outputs()` is called.
9. `build_run_summary(outputs)` returns a dict with at least `run_id`, `num_trials`, `num_success`, `num_failure`.

Minimum success criterion:
- Run 20 trials end-to-end and produce a JSONL artifact with zero missing `TrialOutput` fields.

## Common Student Mistakes and Fixes

1. **Mistake**: An LLM call raises an error inside the batch loop → the entire batch aborts.
   **Fix**: wrap each trial call in `try/except`, log the failure, append an error marker to outputs, and `continue` to the next trial.

2. **Mistake**: `run_id` is not stored in each `TrialOutput` record.
   **Fix**: call `attach_run_metadata()` before every trial invocation; never pass a bare `prompt_package` without run metadata.

3. **Mistake**: Passing the raw LLM provider response directly to the Session 3 parser.
   **Fix**: always call `normalize_subject_response()` → then `extract_subject_text()` before persisting to JSONL.

4. **Mistake**: Hardcoding `model_name` or `temperature` inside the LLM call function.
   **Fix**: always receive `model_name` and `generation_config` as explicit parameters; read them from the config dict.

5. **Mistake**: Mutating the config dict inside `execute_experiment()`.
   **Fix**: make a shallow copy at the start (`cfg = dict(config)`) to avoid unexpected side effects across back-to-back runs.

## Prompt Package Structure

A `prompt_package` is the unit of information sent to the Subject LLM for one trial.
It combines the stimulus content with run-level tracking metadata.

Required keys:

| Key | Type | Source | Example |
|---|---|---|---|
| `stimulus_id` | `str` | from stimulus | `"s0001"` |
| `condition` | `str` | from stimulus | `"active_prime"` |
| `system_prompt` | `str` | built by experimenter | `"You are a helpful language assistant..."` |
| `user_prompt` | `str` | built from prime + target | `"The chef praised the assistant.\n\nDescribe..."` |
| `run_id` | `str` | from config | `"run_001"` |
| `trial_index` | `int` | assigned by runner | `0` |
| `metadata` | `dict` | merged from stimulus + config | `{"structure_label": "active", "prompt_style": "minimal"}` |

Key design decisions:
- `system_prompt` and `user_prompt` are **separate** — this matches the chat API message format and allows swapping instruction styles cleanly.
- `run_id` and `trial_index` are attached by `attach_run_metadata()`, not baked into the stimulus itself.
- `metadata` is a flat, serializable dict — avoid nesting objects inside it.

In [ ]:
# Toy example: building a prompt_package from a stimulus record (for understanding only)

toy_stimulus = {
    "stimulus_id": "s0001",
    "condition": "active_prime",
    "prime_text": "The chef praised the assistant.",
    "target_prompt": "Describe a scene involving a chef and an assistant.",
    "metadata": {"structure_label": "active", "template_name": "active_prime_v1"},
}

# Step 1: build_experimenter_prompt() produces the core package (no run metadata yet)
def toy_build_prompt(stimulus: dict, prompt_style: str = "minimal") -> dict:
    system_prompt = "You are a helpful language assistant. Complete the following scenario naturally."
    if prompt_style == "minimal":
        user_prompt = f"{stimulus['prime_text']}\n\n{stimulus['target_prompt']}"
    else:  # "instructed"
        user_prompt = (
            f"Prime sentence: {stimulus['prime_text']}\n"
            f"Task: {stimulus['target_prompt']}"
        )
    return {
        "stimulus_id": stimulus["stimulus_id"],
        "condition": stimulus["condition"],
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "metadata": dict(stimulus.get("metadata", {})),
    }

# Step 2: attach_run_metadata() adds tracing fields
def toy_attach_metadata(package: dict, run_id: str, trial_index: int) -> dict:
    package = dict(package)
    package["run_id"] = run_id
    package["trial_index"] = trial_index
    return package

prompt_package = toy_build_prompt(toy_stimulus, prompt_style="minimal")
prompt_package = toy_attach_metadata(prompt_package, run_id="run_001", trial_index=0)

print("prompt_package keys: ", list(prompt_package.keys()))
print("user_prompt:         ", prompt_package["user_prompt"])
print("run_id:              ", prompt_package["run_id"])
print("trial_index:         ", prompt_package["trial_index"])

In [ ]:
# Toy TrialOutput example (from src/common/types.py)
from dataclasses import dataclass, asdict
from typing import Any

@dataclass
class TrialOutput:
    """Single trial output emitted by subject LLM."""
    run_id: str
    trial_index: int
    stimulus_id: str
    model_name: str
    response_text: str
    metadata: dict

# What run_single_trial() should return:
toy_trial_output = TrialOutput(
    run_id="run_001",
    trial_index=0,
    stimulus_id="s0001",
    model_name="gpt-4o-mini",
    response_text="The assistant was praised by the chef.",
    metadata={
        "condition": "active_prime",
        "structure_label": "active",
        "prompt_style": "minimal",
        "finish_reason": "stop",
        "usage_tokens": 24,
    },
)

print("TrialOutput fields:     ", list(asdict(toy_trial_output).keys()))
print("response_text:          ", toy_trial_output.response_text)
print("condition from metadata:", toy_trial_output.metadata["condition"])

## Function I/O Cheat Sheet (Session 2)

| Function | Example Input | Example Output | Why this shape |
|---|---|---|---|
| `build_default_config()` | None | `{"run_id": "run_001", "model_name": "gpt-4o-mini", ...}` | One-call setup for notebook experiments |
| `validate_experiment_config(config)` | config dict | `[]` or `["missing field: 'run_id'"]` | Non-throwing; returns all errors in one pass |
| `load_experiment_config(path)` | `"config.yaml"` | config dict | Separates config from code |
| `build_experimenter_prompt(stimulus, prompt_style)` | stimulus dict, `"minimal"` | prompt_package dict (no run metadata yet) | Decouples stimulus content from tracking fields |
| `attach_run_metadata(package, run_id, trial_index)` | package dict, `"run_001"`, `0` | same dict + `run_id` + `trial_index` | Run tracing added in one explicit step |
| `preview_prompt(package)` | prompt_package dict | `"[system] You are...\n[user] The chef..."` | Human-readable check for notebook debugging |
| `invoke_subject_llm(package, model_name, gen_config)` | package, `"gpt-4o-mini"`, `{...}` | raw provider response dict | Provider-agnostic; raw output kept before normalization |
| `normalize_subject_response(raw)` | raw provider dict | `{"text": "...", "finish_reason": "stop", "usage": {...}}` | Isolates provider differences from all downstream code |
| `extract_subject_text(normalized)` | normalized dict | `"The assistant was praised by the chef."` | Clean string ready for Session 3 parser |
| `run_single_trial(stimulus, model_name, gen_config)` | one stimulus dict, model, config | `TrialOutput` dataclass | Atomic experiment unit |
| `run_trial_batch(stimuli, model_name, gen_config)` | list of stimuli, model, config | `list[TrialOutput]` | Batch with per-trial fault tolerance |
| `summarize_run_failures(outputs)` | list of TrialOutput | `{"success": 78, "failure": 2}` | Run health check after batch completes |
| `execute_experiment(config)` | full config dict | run_summary dict | One-call orchestration |
| `persist_run_outputs(outputs, output_dir, run_id)` | outputs, `"artifacts/runs"`, `"run_001"` | `{"jsonl": "artifacts/runs/run_001.jsonl"}` | Durable artifact with predictable paths |
| `build_run_summary(outputs)` | list of TrialOutput | `{"run_id": "run_001", "num_trials": 80, ...}` | Top-level stats for notebook display and Session 3 |

Implementation note:
- `invoke_subject_llm` → `normalize_subject_response` → `extract_subject_text` form a fixed pipeline.
- Never skip normalization — it protects all downstream code from LLM provider API changes.

In [ ]:
# Toy I/O examples for Session 2 functions (for understanding only — no real LLM calls)

# 1) build_default_config() expected return shape
default_config_example = {
    "run_id": "run_001",
    "model_name": "gpt-4o-mini",
    "stimuli_path": "data/processed/stimuli_balanced_seed_42.jsonl",
    "num_trials": 40,
    "seed": 42,
    "prompt_style": "minimal",
    "generation": {"temperature": 0.3, "max_tokens": 120},
}

# 2) validate_experiment_config() return shapes
valid_config_errors_example = []
invalid_config_errors_example = [
    "missing required field: 'stimuli_path'",
    "generation.temperature must be in [0, 1], got 1.5",
]

# 3) normalize_subject_response() expected output shape
normalized_response_example = {
    "text": "The assistant was praised by the chef.",
    "finish_reason": "stop",
    "usage": {"prompt_tokens": 45, "completion_tokens": 12},
}

# 4) run_single_trial() expected TrialOutput shape
trial_output_example = {
    "run_id": "run_001",
    "trial_index": 0,
    "stimulus_id": "s0001",
    "model_name": "gpt-4o-mini",
    "response_text": "The assistant was praised by the chef.",
    "metadata": {
        "condition": "active_prime",
        "structure_label": "active",
        "finish_reason": "stop",
    },
}

# 5) summarize_run_failures() expected output
run_failure_summary_example = {"success": 78, "failure": 2}

# 6) build_run_summary() expected output
run_summary_example = {
    "run_id": "run_001",
    "num_trials": 80,
    "num_success": 78,
    "num_failure": 2,
    "model_name": "gpt-4o-mini",
    "seed": 42,
    "output_path": "artifacts/runs/run_001_seed42.jsonl",
}

print("Default config keys:  ", list(default_config_example.keys()))
print("TrialOutput keys:     ", list(trial_output_example.keys()))
print("Run summary:          ", run_summary_example)

In [ ]:
# Run artifact JSONL format example (one line = one trial)
import json
from pathlib import Path

# One TrialOutput record serialized as a JSONL line
trial_record = {
    "run_id": "run_001",
    "trial_index": 0,
    "stimulus_id": "s0001",
    "model_name": "gpt-4o-mini",
    "response_text": "The assistant was praised by the chef.",
    "metadata": {
        "condition": "active_prime",
        "structure_label": "active",
        "prompt_style": "minimal",
        "finish_reason": "stop",
        "usage_tokens": 24,
    },
}

print("One JSONL line example:")
print(json.dumps(trial_record, ensure_ascii=False))

# Output path convention
seed = 42
run_id = "run_001"
output_path = Path("../artifacts/runs") / f"run_{run_id}_seed{seed}.jsonl"
print("\nSuggested output path:", output_path)

# persist_run_outputs() in src/experiments/runner.py should write this file automatically.
# Each line in the file is one serialized TrialOutput record.

## Run Scale Guidance

| Scale | Num Trials | Purpose |
|---|---|---|
| Smoke test | 10–20 | Verify pipeline runs end-to-end without errors |
| Pilot | 50–100 | Baseline effect size estimate; cost-effective |
| Assignment target | 120–240 | Enough for chi-square and logistic regression in Session 3 |
| Stronger signal | 300–600 | Research-grade statistical power; optional stretch goal |

**Rule**: `num_trials` ≤ number of stimuli in your Session 1 JSONL file.

**Tip**: always run a smoke test (10–20 trials) first to catch integration bugs before scaling up.

Cost note: at 120 trials × ~150 tokens/call ≈ 18,000 tokens total — well within typical free tier limits.

## What Session 2 Is (and Is Not)

Session 2 theme: **multi-agent orchestration + fault-tolerant batch execution + durable artifact persistence**.

This session is **not** asking students to:
- Manually write LLM responses.
- Implement a complex LLM provider SDK from scratch — use an existing library (openai, litellm, etc.).
- Analyze outputs — that is Session 3.

This session **is** asking students to:
1. Implement config loading and validation.
2. Build structured prompt packages from stimulus records.
3. Wrap one LLM provider call in a normalized interface.
4. Run a batch trial loop that survives individual trial failures.
5. Persist structured `TrialOutput` artifacts to `artifacts/runs/`.
6. Return a run summary that Session 3 can read directly.

Why these choices matter:
- **Fault tolerance**: production agent loops always have partial failures; handle them explicitly.
- **Normalization**: isolating provider differences in `normalize_subject_response()` protects the entire downstream analysis pipeline from API changes.
- **Artifact schema**: if `TrialOutput` fields are wrong here, Sessions 3 and 4 will silently produce incorrect analysis.

## Student Tasks

Implement in this dependency order:

1. **`build_default_config()` and `validate_experiment_config()`** — no dependencies; start here.
2. **`build_experimenter_prompt(stimulus, prompt_style)`** — depends on `StimulusRecord` schema from Session 1.
3. **`attach_run_metadata(package, run_id, trial_index)`** — depends on step 2.
4. **`preview_prompt(package)`** — depends on step 2; use for manual debugging in notebook.
5. **`invoke_subject_llm()` and `normalize_subject_response()`** — independent of steps 2–4; use your chosen LLM SDK.
6. **`extract_subject_text(normalized)`** — depends on step 5.
7. **`run_single_trial()`** — depends on steps 2, 3, 5, 6.
8. **`run_trial_batch()` and `summarize_run_failures()`** — depends on step 7.
9. **`persist_run_outputs()` and `build_run_summary()`** — depends on step 8.
10. **`execute_experiment(config)`** — depends on all above; implement last.

In [ ]:
# Full toy multi-agent loop (mock — runs without any real LLM API call)
# Shows how all Session 2 pieces connect end-to-end.
from dataclasses import dataclass, asdict
import json

# --- Mock stimulus records (from Session 1 output) ---
mock_stimuli = [
    {
        "stimulus_id": "s0001",
        "condition": "active_prime",
        "prime_text": "The chef praised the assistant.",
        "target_prompt": "Describe a scene involving a chef and an assistant.",
        "metadata": {"structure_label": "active"},
    },
    {
        "stimulus_id": "s0002",
        "condition": "passive_prime",
        "prime_text": "The assistant was praised by the chef.",
        "target_prompt": "Describe a scene involving a chef and an assistant.",
        "metadata": {"structure_label": "passive"},
    },
    {
        "stimulus_id": "s0003",
        "condition": "active_prime",
        "prime_text": "The manager approved the proposal.",
        "target_prompt": "Describe a scene involving a manager and a proposal.",
        "metadata": {"structure_label": "active"},
    },
]

# --- Mock config ---
mock_config = {
    "run_id": "run_mock",
    "model_name": "mock-llm",
    "stimuli_path": "data/processed/stimuli_balanced_seed_42.jsonl",
    "num_trials": 3,
    "seed": 42,
    "prompt_style": "minimal",
    "generation": {"temperature": 0.3, "max_tokens": 120},
}

# --- Mock experimenter (mirrors src/agents/experimenter.py) ---
class MockExperimenter:
    def build_prompt(self, stimulus, prompt_style="minimal"):
        return {
            "stimulus_id": stimulus["stimulus_id"],
            "condition": stimulus["condition"],
            "system_prompt": "You are a helpful language assistant.",
            "user_prompt": f"{stimulus['prime_text']}\n\n{stimulus['target_prompt']}",
            "metadata": dict(stimulus.get("metadata", {})),
        }

    def attach_metadata(self, package, run_id, trial_index):
        p = dict(package)
        p["run_id"] = run_id
        p["trial_index"] = trial_index
        return p

# --- Mock subject LLM (mirrors src/agents/subject_llm.py) ---
class MockSubjectLLM:
    _responses = {
        "active_prime": "The chef quickly approved the assistant's request.",
        "passive_prime": "The proposal was swiftly approved by the manager.",
    }

    def invoke(self, package, model_name, gen_config):
        # Simulate a raw provider response
        text = self._responses.get(package["condition"], "A scene unfolded naturally.")
        return {
            "choices": [{"message": {"content": text}, "finish_reason": "stop"}],
            "usage": {"total_tokens": 20},
        }

    def normalize(self, raw_response):
        choice = raw_response["choices"][0]
        return {
            "text": choice["message"]["content"],
            "finish_reason": choice["finish_reason"],
            "usage": raw_response.get("usage", {}),
        }

    def extract_text(self, normalized):
        return normalized["text"]

# --- Run the batch trial loop (mirrors src/agents/dialogue_runner.py) ---
experimenter = MockExperimenter()
subject = MockSubjectLLM()
trial_outputs = []

for i, stimulus in enumerate(mock_stimuli[:mock_config["num_trials"]]):
    try:
        package = experimenter.build_prompt(stimulus, mock_config["prompt_style"])
        package = experimenter.attach_metadata(package, mock_config["run_id"], i)
        raw = subject.invoke(package, mock_config["model_name"], mock_config["generation"])
        normalized = subject.normalize(raw)
        response_text = subject.extract_text(normalized)
        output = {
            "run_id": package["run_id"],
            "trial_index": package["trial_index"],
            "stimulus_id": stimulus["stimulus_id"],
            "model_name": mock_config["model_name"],
            "response_text": response_text,
            "metadata": {
                "condition": stimulus["condition"],
                **stimulus.get("metadata", {}),
                "finish_reason": normalized["finish_reason"],
            },
        }
        trial_outputs.append(output)
        print(f"Trial {i}: [{stimulus['condition']:20s}] → {response_text[:55]}")
    except Exception as e:
        print(f"Trial {i}: FAILED — {e}")

# --- Run summary (mirrors src/experiments/runner.py) ---
run_summary = {
    "run_id": mock_config["run_id"],
    "num_trials": mock_config["num_trials"],
    "num_success": len(trial_outputs),
    "num_failure": mock_config["num_trials"] - len(trial_outputs),
    "model_name": mock_config["model_name"],
}
print("\nRun summary:")
print(json.dumps(run_summary, indent=2))

## Using Google Gemini as the Subject LLM

### Step 1: Get an API Key

1. Go to [https://aistudio.google.com](https://aistudio.google.com) and sign in with a Google account.
2. Click **Get API key** in the left sidebar → **Create API key**.
3. Copy the generated key (format: `AIzaSy...`).

### Step 2: Set the Environment Variable

**Do not hardcode the key in source files.** Set it as an environment variable:

```powershell
# PowerShell — current session only
$env:GEMINI_API_KEY = "AIzaSy..."

# PowerShell — persist across sessions
[System.Environment]::SetEnvironmentVariable("GEMINI_API_KEY", "AIzaSy...", "User")
```

### Step 3: Install the SDK

```bash
pip install google-genai
```

### Free Tier Limits (as of 2026)

| Model | Free Limit |
|---|---|
| `gemini-1.5-flash` | 1,500 requests/day · 15 requests/min |
| `gemini-1.5-pro` | 50 requests/day (not recommended for this project) |

Use **`gemini-1.5-flash`** for this project — it is fast and comfortably within the free tier for all session scales.

### Connecting to the Project

When implementing `invoke_subject_llm()` in `src/agents/subject_llm.py`, use `invoke_gemini()` from the code cell below as a reference.
`normalize_subject_response()` converts the raw response into the standard schema so that the Session 3 analysis pipeline has no dependency on any specific LLM provider.

In [ ]:
# Google Gemini — real LLM call example
# Before running, install: pip install google-genai
# And set the environment variable: $env:GEMINI_API_KEY = "your_key_here"

import os
from google import genai
from google.genai import types

# Initialize client (API key is read from environment variable, do not hardcode in code)
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

def invoke_gemini(prompt_package: dict, model_name: str = "gemini-1.5-flash", gen_config: dict = {}) -> dict:
    """Call Gemini API and return raw response dict."""
    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=(
                # Combine system_prompt and user_prompt (Gemini base version does not distinguish system/user)
                f"{prompt_package['system_prompt']}\n\n{prompt_package['user_prompt']}"
            ))]
        )
    ]
    config = types.GenerateContentConfig(
        temperature=gen_config.get("temperature", 0.3),
        max_output_tokens=gen_config.get("max_tokens", 120),
    )
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=contents,
        config=config,
    )
    # 返回统一格式的 raw response，方便 normalize_subject_response() 处理
    return {
        "choices": [{
            "message": {"content": response.text},
            "finish_reason": response.candidates[0].finish_reason.name if response.candidates else "unknown",
        }],
        "usage": {"total_tokens": response.usage_metadata.total_token_count if response.usage_metadata else 0},
    }

def normalize_gemini_response(raw: dict) -> dict:
    """把 raw response 转为标准 schema（与 mock 保持一致）。"""
    choice = raw["choices"][0]
    return {
        "text": choice["message"]["content"],
        "finish_reason": choice["finish_reason"],
        "usage": raw.get("usage", {}),
    }

# --- Quick test (requires valid API key) ---
# test_package = {
#     "system_prompt": "You are a helpful language assistant.",
#     "user_prompt": "The chef praised the assistant.\n\nDescribe a scene involving a chef and an assistant.",
#     "stimulus_id": "s0001",
#     "condition": "active_prime",
# }
# raw = invoke_gemini(test_package, model_name="gemini-1.5-flash")
# normalized = normalize_gemini_response(raw)
# print("Response:", normalized["text"])
# print("Finish reason:", normalized["finish_reason"])
# print("Tokens used:", normalized["usage"])

In [ ]:
# Verification checks — run after implementing TODOs in src/
# Uncomment the two lines below after execute_experiment() is implemented:
# from src.experiments.runner import execute_experiment
# run_summary = execute_experiment(example_config)

# For now, use a mock summary to validate assertion logic
run_summary = {
    "run_id": "run_001",
    "num_trials": 20,
    "num_success": 19,
    "num_failure": 1,
    "model_name": "gpt-4o-mini",
    "output_path": "artifacts/runs/run_001_seed42.jsonl",
}

# 1. Check required keys exist in run_summary
required_summary_keys = {"run_id", "num_trials", "num_success", "num_failure"}
assert required_summary_keys.issubset(run_summary.keys()), \
    f"Missing keys: {required_summary_keys - run_summary.keys()}"

# 2. Check counts are consistent
assert run_summary["num_success"] + run_summary["num_failure"] == run_summary["num_trials"], \
    "Success + failure count mismatch"

# 3. Check failure rate is acceptable (< 10%)
failure_rate = run_summary["num_failure"] / run_summary["num_trials"]
assert failure_rate < 0.10, f"Failure rate too high: {failure_rate:.1%}"

# 4. (After implementation) Check output file exists
# from pathlib import Path
# output_path = Path("..") / run_summary["output_path"]
# assert output_path.exists(), f"Output file not found: {output_path}"

# 5. (After implementation) Spot-check 3 random records from output file
# import random, json
# with open(output_path) as f:
#     records = [json.loads(line) for line in f]
# required_trial_keys = {"run_id", "trial_index", "stimulus_id", "model_name", "response_text", "metadata"}
# for rec in random.sample(records, min(3, len(records))):
#     missing = required_trial_keys - rec.keys()
#     assert not missing, f"Record missing fields: {missing}"

print("All assertions passed.")
print(f"Failure rate: {failure_rate:.1%}")
print(f"Success: {run_summary['num_success']} / {run_summary['num_trials']} trials")

## Reflection Prompt

What control variables besides temperature might influence measured priming strength?